In [1]:
import warnings
from pathlib import Path
from math import exp, factorial
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)





In [2]:
# ----------------------------------------------------
# 1. KONFIGURASI DAN METRIK (15x15 LOSS TENSOR)
DATA_DIR  = Path('.')
train_df  = pd.read_csv(DATA_DIR / 'train.csv')
test_df   = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv') if (DATA_DIR / 'sample_submission.csv').exists() else pd.read_csv(DATA_DIR / 'sample submission.csv')

train_df['date'] = pd.to_datetime(train_df['date'], errors='coerce')
test_df['date']  = pd.to_datetime(test_df['date'],  errors='coerce')
MODERN_CUTOFF = pd.Timestamp('2000-01-01')

MAX_G = 15 # Diperbesar dari 8 ke 15 untuk menangkap "Long Tail" Probabilitas
MAX_PRED_GOALS = MAX_G + 1  
MAX_TRUE_GOALS = 25 
_LOSS_TENSOR = np.zeros((MAX_PRED_GOALS, MAX_PRED_GOALS, MAX_TRUE_GOALS, MAX_TRUE_GOALS))

def tourn_weight(name):
    if pd.isna(name): return 1.20
    x = str(name).lower()
    if 'friendly' in x: return 0.96
    if 'world cup' in x and 'qualif' not in x: return 2.00
    if any(k in x for k in ['championship','nations cup','copa america','euro ','gold cup','nations league']): return 1.80
    if any(k in x for k in ['qualif','qualifier','qualification']): return 1.60
    return 1.20

for pred_t in range(MAX_PRED_GOALS):
    for pred_o in range(MAX_PRED_GOALS):
        for true_t in range(MAX_TRUE_GOALS):
            for true_o in range(MAX_TRUE_GOALS):
                mae = (abs(true_t - pred_t) + abs(true_o - pred_o)) / 2.0
                exact_pen = 0.30 if (pred_t != true_t or pred_o != true_o) else 0.0
                pred_out = 1 if pred_t > pred_o else (0 if pred_t == pred_o else -1)
                true_out = 1 if true_t > true_o else (0 if true_t == true_o else -1)
                outcome_pen = 0.25 if pred_out != true_out else 0.0
                pred_gd = pred_t - pred_o
                true_gd = true_t - true_o
                gd_pen = 0.15 if pred_gd != true_gd else 0.0
                total_penalty = exact_pen + outcome_pen + gd_pen
                multiplier = 1.5 if pred_out != true_out else 1.0
                _LOSS_TENSOR[pred_t, pred_o, true_t, true_o] = ((mae + total_penalty) * multiplier) ** 1.5


In [3]:
# ----------------------------------------------------
# 2. DIXON-COLES BIVARIATE & EXPECTED LOSS DECODER

def dixon_coles_matrix(lam_t, lam_o, rho, max_g=MAX_G):
    vals = np.arange(max_g+1)
    p_t = np.array([exp(-lam_t)*(lam_t**k)/factorial(k) for k in vals])
    p_o = np.array([exp(-lam_o)*(lam_o**k)/factorial(k) for k in vals])
    matrix = np.outer(p_t, p_o)
    
    tau_00 = max(0.0, 1.0 - lam_t * lam_o * rho)
    tau_01 = max(0.0, 1.0 + lam_t * rho)
    tau_10 = max(0.0, 1.0 + lam_o * rho)
    tau_11 = max(0.0, 1.0 - rho)
    
    matrix[0, 0] *= tau_00
    matrix[0, 1] *= tau_01
    matrix[1, 0] *= tau_10
    matrix[1, 1] *= tau_11
    
    sum_prob = np.sum(matrix)
    if sum_prob > 0: matrix /= sum_prob
    return matrix

def decode_awmae_expected_loss(prob_matrix, max_g=MAX_G):
    # Mengalikan prob_matrix dengan _LOSS_TENSOR ukuran 16x16
    expected_losses = np.sum(_LOSS_TENSOR[:, :, :max_g+1, :max_g+1] * prob_matrix, axis=(2, 3))
    best_idx = np.unravel_index(np.argmin(expected_losses, axis=None), expected_losses.shape)
    return best_idx[0], best_idx[1]


In [4]:

# ----------------------------------------------------
# 3. DIXON-MAHER ITERATIVE POISSON ENGINE

def fit_dixon_maher(df, halflife_years=2.0, n_iter=50):
    ratings = {}
    ref = df['date'].max()
    alpha = np.log(2) / (halflife_years * 365.25)

    for gender in df['gender'].unique():
        gdf = df[df['gender'] == gender].copy().reset_index(drop=True)
        if len(gdf) < 20: continue
        days = (ref - gdf['date']).dt.days.clip(lower=0).values
        w = np.exp(-alpha * days)
        goals_arr  = gdf['team_goals'].values.astype(float)
        team_arr   = gdf['team'].values
        opp_arr    = gdf['opponent'].values
        home_arr   = gdf['is_home'].values
        neut_arr   = gdf['neutral'].values
        teams = sorted(set(gdf['team'].tolist()) | set(gdf['opponent'].tolist()))
        h_mask = home_arr == 1; n_mask = neut_arr == 1; a_mask = (~h_mask) & (~n_mask)
        avg_all  = np.average(goals_arr, weights=w)
        avg_h    = np.average(goals_arr[h_mask], weights=w[h_mask]) if h_mask.sum()>0 else avg_all
        avg_a    = np.average(goals_arr[a_mask], weights=w[a_mask]) if a_mask.sum()>0 else avg_all
        avg_n    = np.average(goals_arr[n_mask], weights=w[n_mask]) if n_mask.sum()>0 else avg_all
        h_home, h_away, h_neutral = avg_h / avg_all, avg_a / avg_all, avg_n / avg_all
        h_arr = np.where(h_mask, h_home, np.where(n_mask, h_neutral, h_away))
        attack  = {t: 1.0 for t in teams}; defense = {t: 1.0 for t in teams}; mu = float(avg_all)

        for _ in range(n_iter):
            for t in teams:
                mask = team_arr == t
                if mask.sum() == 0: continue
                d_opp = np.array([defense.get(o, 1.0) for o in opp_arr[mask]])
                num = np.sum(w[mask] * goals_arr[mask])
                den = np.sum(w[mask] * mu * d_opp * h_arr[mask])
                attack[t] = num / max(den, 1e-9)
            for t in teams:
                mask = opp_arr == t
                if mask.sum() == 0: continue
                a_team = np.array([attack.get(tm, 1.0) for tm in team_arr[mask]])
                num = np.sum(w[mask] * goals_arr[mask])
                den = np.sum(w[mask] * mu * a_team * h_arr[mask])
                defense[t] = num / max(den, 1e-9)
            att_m = np.mean(list(attack.values())); def_m = np.mean(list(defense.values()))
            mu *= att_m * def_m
            for t in teams: attack[t] /= att_m; defense[t] /= def_m

        conf_map = dict(zip(gdf['team'], gdf['confederation_team']))
        att_conf = {c: float(np.mean([attack[t] for t in teams if conf_map.get(t) == c])) for c in set(conf_map.values())}
        def_conf = {c: float(np.mean([defense[t] for t in teams if conf_map.get(t) == c])) for c in set(conf_map.values())}
        ratings[gender] = dict(attack=attack, defense=defense, mu=mu, h_home=h_home, h_away=h_away, h_neutral=h_neutral, att_conf=att_conf, def_conf=def_conf)
    return ratings

def compute_lambda(df, ratings):
    lam_t, lam_o = np.ones(len(df)), np.ones(len(df))
    for gender, r in ratings.items():
        mask = (df['gender'] == gender).values
        if mask.sum() == 0: continue
        gdf = df[mask]
        get_att = np.vectorize(lambda t, c: r['attack'].get(t, r['att_conf'].get(c, 1.0)))
        get_def = np.vectorize(lambda t, c: r['defense'].get(t, r['def_conf'].get(c, 1.0)))
        att_t = get_att(gdf['team'], gdf['confederation_team'])
        def_o = get_def(gdf['opponent'], gdf['confederation_opp'])
        att_o = get_att(gdf['opponent'], gdf['confederation_opp'])
        def_t = get_def(gdf['team'], gdf['confederation_team'])
        is_h, is_n = gdf['is_home'].values, gdf['neutral'].values
        h_t = np.where(is_n==1, r['h_neutral'], np.where(is_h==1, r['h_home'], r['h_away']))
        h_o = np.where(is_n==1, r['h_neutral'], np.where(is_h==1, r['h_away'], r['h_home']))
        lam_t[mask] = r['mu'] * att_t * def_o * h_t
        lam_o[mask] = r['mu'] * att_o * def_t * h_o
    return np.clip(lam_t, 0.05, 15), np.clip(lam_o, 0.05, 15)

def prepare_features(df, lam_t, lam_o):
    df_out = df.copy()
    df_out['lam_team'], df_out['lam_opp'] = lam_t, lam_o
    df_out['lam_diff'] = lam_t - lam_o
    df_out['year'] = df_out['date'].dt.year.fillna(2023)
    df_out['tourn_tier'] = df_out['tournament'].apply(tourn_weight)
    
    cat_feats = ['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']
    num_feats = ['is_home', 'neutral', 'population_team', 'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp', 'altitude_venue', 'distance_travel_team', 'distance_travel_opp', 'temperature_venue']
    
    df_cb = df_out.copy()
    for c in cat_feats: df_cb[c] = df_cb[c].fillna('Unknown').astype(str)
    for c in num_feats: df_cb[c] = pd.to_numeric(df_cb[c], errors='coerce').fillna(-999)
    
    df_tree = df_out.copy()
    for c in cat_feats:
        global_cats = pd.unique(pd.concat([train_df[c], test_df[c]]).fillna('Unknown').astype(str))
        df_tree[c] = pd.Categorical(df_tree[c].fillna('Unknown').astype(str), categories=global_cats)
    for c in num_feats: df_tree[c] = pd.to_numeric(df_tree[c], errors='coerce').fillna(-999)
    
    return df_cb, df_tree, cat_feats, num_feats

def apply_mercy_rule(lam):
    # Jika model menebak > 2.5, susutkan kelebihannya untuk memangkas resiko Over-Prediction
    return np.where(lam > 2.5, 2.5 + (lam - 2.5) * 0.6, lam)


In [5]:
# ----------------------------------------------------
# 4. TRAINING 3 MODEL
print("1. Melatih Base Dixon-Maher...")
prior_full = train_df[train_df['date'] >= MODERN_CUTOFF]
ratings_full = fit_dixon_maher(prior_full, halflife_years=2.0, n_iter=50)

lam_train_t, lam_train_o = compute_lambda(train_df, ratings_full)
lam_test_t, lam_test_o = compute_lambda(test_df, ratings_full)

train_cb, train_tree, CAT_FEATS, NUM_FEATS = prepare_features(train_df, lam_train_t, lam_train_o)
test_cb, test_tree, _, _ = prepare_features(test_df, lam_test_t, lam_test_o)

feat_cols = CAT_FEATS + NUM_FEATS + ['lam_team', 'lam_opp', 'lam_diff', 'year', 'tourn_tier']
cat_idx = [feat_cols.index(c) for c in CAT_FEATS]
w_train = train_df['tournament'].apply(tourn_weight).values

print("2. Melatih The Holy Trinity (CatBoost, LightGBM, XGBoost)...")
# --- Model 1: CatBoost ---
cb_t = CatBoostRegressor(loss_function='Poisson', depth=5, learning_rate=0.05, iterations=600, verbose=0, random_seed=42)
cb_t.fit(train_cb[feat_cols], train_df['team_goals'].fillna(0).values, cat_features=cat_idx, sample_weight=w_train)
cb_o = CatBoostRegressor(loss_function='Poisson', depth=5, learning_rate=0.05, iterations=600, verbose=0, random_seed=43)
cb_o.fit(train_cb[feat_cols], train_df['opp_goals'].fillna(0).values, cat_features=cat_idx, sample_weight=w_train)

cb_pred_t = np.clip(cb_t.predict(test_cb[feat_cols]), 0.05, 15)
cb_pred_o = np.clip(cb_o.predict(test_cb[feat_cols]), 0.05, 15)

# --- Model 2: LightGBM ---
lgb_params = {'objective':'poisson', 'max_depth':5, 'learning_rate':0.05, 'n_estimators':600, 'random_state':42, 'verbose': -1}
lgb_t = LGBMRegressor(**lgb_params)
lgb_t.fit(train_tree[feat_cols], train_df['team_goals'].fillna(0).values, sample_weight=w_train)
lgb_o = LGBMRegressor(**{**lgb_params, 'random_state':43})
lgb_o.fit(train_tree[feat_cols], train_df['opp_goals'].fillna(0).values, sample_weight=w_train)

lgb_pred_t = np.clip(lgb_t.predict(test_tree[feat_cols]), 0.05, 15)
lgb_pred_o = np.clip(lgb_o.predict(test_tree[feat_cols]), 0.05, 15)

# --- Model 3: XGBoost ---
xgb_params = {'objective':'count:poisson', 'max_depth':5, 'learning_rate':0.05, 'n_estimators':600, 'random_state':42, 'enable_categorical':True}
xgb_t = XGBRegressor(**xgb_params)
xgb_t.fit(train_tree[feat_cols], train_df['team_goals'].fillna(0).values, sample_weight=w_train)
xgb_o = XGBRegressor(**{**xgb_params, 'random_state':43})
xgb_o.fit(train_tree[feat_cols], train_df['opp_goals'].fillna(0).values, sample_weight=w_train)

xgb_pred_t = np.clip(xgb_t.predict(test_tree[feat_cols]), 0.05, 15)
xgb_pred_o = np.clip(xgb_o.predict(test_tree[feat_cols]), 0.05, 15)

print("3. Mengeksekusi Lambda Ensembling & Mercy Rule Dampener...")
# Lambda Ensembling sesuai Blueprint
raw_lam_t = (0.4 * lgb_pred_t) + (0.4 * cb_pred_t) + (0.2 * xgb_pred_t)
raw_lam_o = (0.4 * lgb_pred_o) + (0.4 * cb_pred_o) + (0.2 * xgb_pred_o)

# Dampener Mencegah Over-Prediksi
damp_lam_t = apply_mercy_rule(raw_lam_t)
damp_lam_o = apply_mercy_rule(raw_lam_o)

1. Melatih Base Dixon-Maher...
2. Melatih The Holy Trinity (CatBoost, LightGBM, XGBoost)...
3. Mengeksekusi Lambda Ensembling & Mercy Rule Dampener...


In [6]:
# ----------------------------------------------------
# 5. DYNAMIC DIXON-COLES DECODING DENGAN SIMETRI
# ----------------------------------------------------
print("4. Mendekode Matriks (Dynamic Rho & Symmetry)...")
test_df['match_id'] = test_df['Id'].str.split('_').str[0]
final_pred_team = np.zeros(len(test_df), dtype=int)
final_pred_opp = np.zeros(len(test_df), dtype=int)

for m_id, idxs in test_df.groupby('match_id').groups.items():
    idx_list = list(idxs)
    if len(idx_list) == 2:
        idx_A, idx_B = idx_list[0], idx_list[1]
        
        # Simetri Pre-Matrix
        sym_lam_t = (damp_lam_t[idx_A] + damp_lam_o[idx_B]) / 2.0
        sym_lam_o = (damp_lam_o[idx_A] + damp_lam_t[idx_B]) / 2.0
        
        # Dynamic Rho Evaluator
        delta_lam = abs(sym_lam_t - sym_lam_o)
        current_rho = -0.08 if delta_lam < 0.5 else 0.0
        
        prob_matrix_A = dixon_coles_matrix(sym_lam_t, sym_lam_o, rho=current_rho)
        p_t_A, p_o_A = decode_awmae_expected_loss(prob_matrix_A)
        
        final_pred_team[idx_A], final_pred_opp[idx_A] = p_t_A, p_o_A
        final_pred_team[idx_B], final_pred_opp[idx_B] = p_o_A, p_t_A
    else:
        for i in idx_list:
            delta_lam = abs(damp_lam_t[i] - damp_lam_o[i])
            current_rho = -0.08 if delta_lam < 0.5 else 0.0
            mat = dixon_coles_matrix(damp_lam_t[i], damp_lam_o[i], rho=current_rho)
            final_pred_team[i], final_pred_opp[i] = decode_awmae_expected_loss(mat)

pred_df = pd.DataFrame({'Id': test_df['Id'], 'team_goals': final_pred_team, 'opp_goals': final_pred_opp})
final_submission = sample_sub[['Id']].merge(pred_df, on='Id', how='left')

OUT_PATH = '04_SubmisiFinal.csv'
final_submission.to_csv(OUT_PATH, index=False)
print(f"DONE! Hasil disimpan ke '{OUT_PATH}'")


4. Mendekode Matriks (Dynamic Rho & Symmetry)...
DONE! Hasil disimpan ke '04_SubmisiFinal.csv'
